In [2]:
import os
import random

import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
from mlflow.data.http_dataset_source import HTTPDatasetSource
from PIL import Image

In [3]:
mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("AgroDiagnosisEDA")

train_dir = (
    "../data/New Plant Diseases Dataset(Augmented)/New Plant Diseases Dataset(Augmented)/train"
)
valid_dir = train_dir.replace("train", "valid")

## Дисбаланс классов

In [4]:
classes = sorted([d for d in os.listdir(train_dir) if os.path.isdir(os.path.join(train_dir, d))])

train_counts = [len(os.listdir(os.path.join(train_dir, c))) for c in classes]
valid_counts = (
    [len(os.listdir(os.path.join(valid_dir, c))) for c in classes]
    if os.path.exists(valid_dir)
    else [0] * len(classes)
)

df_stats = pd.DataFrame(
    {"class": classes, "train_count": train_counts, "valid_count": valid_counts}
).sort_values("train_count", ascending=True)

print(f"Найдено классов: {len(classes)}")
print(f"Изображений в train: {sum(train_counts)}, в valid: {sum(valid_counts)}")
display(df_stats.tail())

Найдено классов: 38
Изображений в train: 70295, в valid: 17572


,class,train_count,valid_count
19,"Pepper,_bell___healthy",1988,497
3,Apple___healthy,2008,502
15,Orange___Haunglongbing_(Citrus_greening),2010,503
0,Apple___Apple_scab,2016,504
24,Soybean___healthy,2022,505


## Структура, яркость, размеры

In [6]:
sample_size = 20
formats, widths, heights, brightness_values = [], [], [], []

for cls in classes:
    class_path = os.path.join(train_dir, cls)
    all_images = os.listdir(class_path)
    sampled = random.sample(all_images, min(sample_size, len(all_images)))

    for img_name in sampled:
        img_path = os.path.join(class_path, img_name)
        formats.append(img_name.split(".")[-1].lower())  # форматы
        try:
            with Image.open(img_path) as img:
                w, h = img.size
                widths.append(w)
                heights.append(h)

                # средняя яркость пикселей
                brightness_values.append(np.mean(img.convert("L")))
        except OSError:
            pass

In [9]:
text = f"""EDA:
1. Минимальное количество картинок в классе - {df_stats["train_count"].min()},
    максимальное - {df_stats["train_count"].max()}.
2. Форматы: {set(formats)}.
3. Разрешения: от {min(widths)}x{min(heights)} до {max(widths)}x{max(heights)}.
4. Яркость: средняя {np.mean(brightness_values):.2f}.
"""

In [10]:
with mlflow.start_run(run_name="eda"):
    kaggle_source = HTTPDatasetSource(
        url="https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset"
    )
    dataset = mlflow.data.from_pandas(
        df_stats, name="Plant Diseases Augmented v2", source=kaggle_source
    )
    mlflow.log_input(dataset, context="eda")

    fig_classes, ax1 = plt.subplots(figsize=(10, 6))
    ax1.barh(df_stats["class"][-15:], df_stats["train_count"][-15:], color="#2b5c8f")
    ax1.set_title("Распределение изображений (Топ-15 классов в Train)")
    ax1.set_xlabel("Количество")
    plt.tight_layout()
    mlflow.log_figure(fig_classes, "eda_class_distribution.png")
    plt.close(fig_classes)

    fig_bright, ax2 = plt.subplots(figsize=(8, 5))
    ax2.hist(brightness_values, bins=30, color="#8f2b5c", alpha=0.8)
    ax2.set_title("Распределение средней яркости изображений")
    ax2.set_xlabel("Яркость")
    ax2.set_ylabel("Частота")
    plt.tight_layout()
    mlflow.log_figure(fig_bright, "eda_brightness_anomalies.png")
    plt.close(fig_bright)

    with open("eda.txt", "w", encoding="utf-8") as f:
        f.write(text)
    mlflow.log_artifact("eda.txt")

    mlflow.log_params(
        {
            "total_classes": len(classes),
            "total_images": sum(train_counts) + sum(valid_counts),
            "mean_brightness": np.mean(brightness_values),
        }
    )

C:\Users\shaki\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(


🏃 View run eda at: http://localhost:5000/#/experiments/5/runs/a0ca572ec1774d10a3a10c585a653236
🧪 View experiment at: http://localhost:5000/#/experiments/5
